# V3 — Auditoria exploratória de alometria: árvores de Caxias do Sul

**V3 separada dos notebooks original e V2.** Testa a plausibilidade *numérica* dos índices C1–C6 diante de faixas de porte da arborização local. Não mede a altura das árvores.

Fontes: [PDAU municipal](https://caxias.rs.gov.br/servicos/meio-ambiente/plano-diretor-de-arborizacao-urbana-pdau) e [Anexo 1](https://gcpstorage.caxias.rs.gov.br/documents/2020/09/c2d2ec2b-dfb4-4646-aadd-01eced79a722.pdf).


## 1. Preparação no Colab

Utiliza somente os arquivos já existentes na pasta Area1. Ajuste `AREA1` se o caminho no Drive for diferente. Não sobrescreve entradas.


In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Fora do Google Colab: informe o caminho local.")
AREA1=Path("/content/drive/MyDrive/Area1")
SAIDA=Path("/content/v3_saida_alometria")
REPO=Path("/content/mestrado-arvores-urbanas")
if not REPO.exists():
    subprocess.run(["git","clone","--depth","1",
      "https://github.com/BrenoBragaGalvao/mestrado-arvores-urbanas.git",str(REPO)],check=True)
SCRIPT=REPO/"experimentos"/"v3_alometria_caxias.py"
for module in ["numpy","pandas","rasterio","geopandas","skimage","matplotlib"]:
    if importlib.util.find_spec(module) is None:
        pipname="scikit-image" if module=="skimage" else module
        subprocess.run([sys.executable,"-m","pip","install","-q",pipname],check=True)
assert (AREA1/"AOI_RGB_NEW.tif").is_file()
assert (AREA1/"Arvores.gpkg").is_file()


## 2. Verificar o que está disponível

Os pontos existentes em `Arvores.gpkg` identificam árvores, **mas não a espécie ou a altura medida de cada uma**. O PDAU é usado apenas como contexto.


In [ ]:
import geopandas as gpd
g=gpd.read_file(AREA1/"Arvores.gpkg")
print("Pontos:",len(g),"CRS:",g.crs,"Campos:",list(g.columns))


## 3. Executar segmentação local e índices de porte

O Quickshift gera **superpixels**, não copas completas. Área do segmento e altura real são conceitos distintos. C3 é um **índice empírico**, não uma medida em metros.


In [ ]:
SAIDA.mkdir(exist_ok=True,parents=True)
subprocess.run([sys.executable,str(SCRIPT),"--area",str(AREA1),
     "--out",str(SAIDA),"--tile","256"],check=True)


## 4. Consultar os resultados e as faixas regionais

As faixas da prefeitura são **por espécie**, e não podem ser atribuídas aos 257 pontos sem identificação botânica.


In [ ]:
import pandas as pd,json
from IPython.display import display,Image
r=json.loads((SAIDA/"v3_resumo.json").read_text(encoding="utf-8"))
display(pd.DataFrame([r]))
display(pd.read_csv(SAIDA/"v3_referencias_alturas_caxias.csv"))
for f in ["v3_histograma_indice_C3.png","v3_sensibilidade_C1_C6.png",
          "v3_faixas_especies_caxias.png"]:
    display(Image(filename=str(SAIDA/f)))


## 5. Sensibilidade adicional (opcional)

Alterar a janela para 512 px pode alterar os superpixels e o índice. Esse teste **não valida a área real da copa**.


In [ ]:
EXECUTAR_512=False
if EXECUTAR_512:
    subprocess.run([sys.executable,str(SCRIPT),"--area",str(AREA1),
        "--out",str(SAIDA.parent/"v3_saida_512"),"--tile","512"],check=True)


## 6. Interpretação científica

**Podemos** relatar a distribuição numérica e a sensibilidade dos índices e consultar alturas possíveis de espécies da região.

**Não podemos** afirmar altura real, espécie individual, acurácia, MAE ou RMSE sem medidas de campo ou validação independente das **mesmas árvores**. Tampouco inferir risco elétrico de distâncias a postes.
